In [1]:
%pip install -q sentence-transformers

import os
import re
import glob
import numpy as np

print("✅ Resume Q&A project setup ready!")

✅ Resume Q&A project setup ready!


In [2]:
NOTES = {
    "knowledge/resume/education.md": """---
type: Resume
title: Education
---
# Education
## Name
My name is Akula Hema Priya.
## Degree
I am pursuing B.Tech in Computer Science and Artificial Intelligence (CAI).
## College
I study at Kakinada Institute of Engineering and Technologies (KIET).
## Diploma
I completed my Diploma with 83.75%.
""",

    "knowledge/resume/projects.md": """---
type: Project
title: Insurance Fraud Claim Scout
---
# Insurance Fraud Claim Scout
## Purpose
The project flags potentially suspicious insurance claims for review.
## Technologies
The project uses Java and Python.
## Java Role
Java processes claims and applies rules based on previous claims,
claim amount, and policy average.
## Python Role
Python calculates a ratio-based fraud risk score.
## Data Files
The project uses claims.csv and fraud_scores.csv.
"""
}

for path, text in NOTES.items():
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as file:
        file.write(text)
    print("Saved:", path)

print("\n✅ Resume notes ready!")

Saved: knowledge/resume/education.md
Saved: knowledge/resume/projects.md

✅ Resume notes ready!


In [3]:
files = [
    path
    for path in sorted(glob.glob("knowledge/**/*.md", recursive=True))
    if os.path.basename(path) not in ("index.md", "log.md")
]

problems = 0

for path in files:
    with open(path, "r", encoding="utf-8") as file:
        text = file.read()

    missing = []

    if not re.search(r"^type:\s*\S+", text, re.M):
        missing.append("type")
    if not re.search(r"^# .+", text, re.M):
        missing.append("title")
    if not re.search(r"^## .+", text, re.M):
        missing.append("section")

    if missing:
        problems += 1
        print("❌", path, "Missing:", ", ".join(missing))
    else:
        print("✅ OK:", path)

index_text = """---
type: Index
title: Resume Knowledge Index
---
# Resume Knowledge Index
"""

for path in files:
    index_text += f"- {path.removeprefix('knowledge/')}\n"

with open("knowledge/index.md", "w", encoding="utf-8") as file:
    file.write(index_text)

print(f"\nChecked {len(files)} notes.")
print("✅ All notes are good!" if problems == 0
      else f"❌ Fix {problems} note(s) before continuing.")

✅ OK: knowledge/resume/education.md
✅ OK: knowledge/resume/projects.md

Checked 2 notes.
✅ All notes are good!


In [11]:
from sentence_transformers import SentenceTransformer

def load_pieces():
    pieces = []

    for path in files:
        with open(path, "r", encoding="utf-8") as file:
            text = file.read()

        title = re.search(r"^# (.+)$", text, re.M).group(1)

        sections = re.split(r"^## ", text, flags=re.M)[1:]

        for section in sections:
            pieces.append({
                "file": path,
                "text": f"{title}: {section.strip()}"
            })

    return pieces

PIECES = load_pieces()

if not PIECES:
    raise ValueError("No resume sections found. Check Cells 2 and 3.")

model = SentenceTransformer("all-MiniLM-L6-v2")

VECTORS = model.encode(
    [piece["text"] for piece in PIECES],
    normalize_embeddings=True
)

def find(question):
    question_vector = model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    scores = VECTORS @ question_vector
    best_index = int(np.argmax(scores))

    return float(scores[best_index]), PIECES[best_index]

print("✅ Search ready!")
print("Total sections:", len(PIECES))

score, piece = find("What is my diploma percentage?")

print("\nFound in:", piece["file"])
print("Matching score:", round(score, 2))
print("Information:", piece["text"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Search ready!
Total sections: 9

Found in: knowledge/resume/education.md
Matching score: 0.86
Information: Education: Diploma
I completed my Diploma with 83.75%.


In [12]:
LIMIT = 0.30

def resume_agent(question):
    if not question.strip():
        print("Please type a question.")
        return

    score, piece = find(question)

    print("QUESTION:", question)

    if score < LIMIT:
        print("ANSWER: Related information dorakaledhu.")
    else:
        print("SOURCE:", piece["file"])
        print("SIMILARITY:", round(score, 2))
        print("ANSWER:", piece["text"])

    print("-" * 45)

resume_agent("What is my name?")
resume_agent("Where do I study?")
resume_agent("What does Python do in my insurance project?")

QUESTION: What is my name?
SOURCE: knowledge/resume/education.md
SIMILARITY: 0.44
ANSWER: Education: Name
My name is Akula Hema Priya.
---------------------------------------------
QUESTION: Where do I study?
SOURCE: knowledge/resume/education.md
SIMILARITY: 0.48
ANSWER: Education: College
I study at Kakinada Institute of Engineering and Technologies (KIET).
---------------------------------------------
QUESTION: What does Python do in my insurance project?
SOURCE: knowledge/resume/projects.md
SIMILARITY: 0.7
ANSWER: Insurance Fraud Claim Scout: Python Role
Python's role in this project is to calculate a ratio-based fraud risk score for insurance claims.
---------------------------------------------


In [14]:
TESTS = [
    ("What is my name?", "## Name", "Akula Hema Priya"),
    ("What degree am I pursuing?", "## Degree", "B.Tech"),
    ("Where do I study?", "## College", "KIET"),
    ("What is my diploma percentage?", "## Diploma", "83.75%"),
    ("What is the purpose of my insurance project?",
     "## Purpose", "suspicious insurance claims"),
    ("Which technologies does my insurance project use?",
     "## Technologies", "Java and Python"),
    ("What does Java do in my project?",
     "## Java Role", "Java processes claims"),
    ("What does Python do in my project?",
     "## Python Role", "ratio-based fraud risk score"),
    ("Which data files does my project use?",
     "## Data Files", "claims.csv"),
]

passed = 0

for question, section, expected_text in TESTS:
    score, piece = find(question)

    correct = (
        score >= LIMIT
        and expected_text.lower() in piece["text"].lower()
    )

    passed += int(correct)
    print(f"{'✅ PASS' if correct else '❌ FAIL'}: {question}")

    if not correct:
        print("  Expected section:", section)
        print("  Retrieved:", piece["text"])
        print("  Similarity:", round(score, 2))

print(f"\nRESULT: {passed}/{len(TESTS)} passed")

✅ PASS: What is my name?
✅ PASS: What degree am I pursuing?
✅ PASS: Where do I study?
✅ PASS: What is my diploma percentage?
✅ PASS: What is the purpose of my insurance project?
✅ PASS: Which technologies does my insurance project use?
✅ PASS: What does Java do in my project?
✅ PASS: What does Python do in my project?
✅ PASS: Which data files does my project use?

RESULT: 9/9 passed


In [15]:
path = "knowledge/resume/projects.md"

with open(path, "r", encoding="utf-8") as file:
    text = file.read()

text = text.replace(
    "Python calculates a ratio-based fraud risk score.",
    "Python's role in this project is to calculate a ratio-based "
    "fraud risk score for insurance claims."
)

text = text.replace(
    "The project uses claims.csv and fraud_scores.csv.",
    "The data files used by this project are claims.csv "
    "and fraud_scores.csv."
)

with open(path, "w", encoding="utf-8") as file:
    file.write(text)

print("✅ Project descriptions updated!")


✅ Project descriptions updated!


In [16]:
import shutil
from google.colab import files

shutil.make_archive(
    "Resume_QA_Notes",
    "zip",
    "knowledge"
)

files.download("Resume_QA_Notes.zip")

print("✅ Resume notes ZIP ready!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Resume notes ZIP ready!


In [17]:
%pip install -q gradio

import gradio as gr

def ask_resume(question):
    if not question.strip():
        return "", "", "Please type a question."

    score, piece = find(question)

    if score < LIMIT:
        return (
            "No matching source",
            f"{score:.3f}",
            "Related information dorakaledhu."
        )

    return (
        piece["file"],
        f"{score:.3f}",
        piece["text"]
    )

with gr.Blocks(title="Hema Priya Resume Q&A") as demo:
    gr.Markdown(
        "# 📄 Hema Priya Resume Q&A\n"
        "Ask about my education and project. "
        "Answers show the matching resume note."
    )

    question = gr.Textbox(
        label="Your question",
        placeholder="What does Python do in my project?"
    )

    ask_button = gr.Button("Ask", variant="primary")

    source = gr.Textbox(label="Source")
    similarity = gr.Textbox(label="Similarity score")
    answer = gr.Textbox(label="Answer", lines=5)

    gr.Examples(
        examples=[
            ["What is my name?"],
            ["Where do I study?"],
            ["What is my diploma percentage?"],
            ["What does Python do in my project?"],
            ["Which data files does my project use?"]
        ],
        inputs=[question]
    )

    ask_button.click(
        fn=ask_resume,
        inputs=question,
        outputs=[source, similarity, answer]
    )

    question.submit(
        fn=ask_resume,
        inputs=question,
        outputs=[source, similarity, answer]
    )

demo.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [18]:
%pip install -q requests beautifulsoup4

import requests
from bs4 import BeautifulSoup

url = "https://hemapriya-dev.github.io/hema-priya-portfolio/"

response = requests.get(url, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

for tag in soup(["script", "style", "nav", "footer"]):
    tag.decompose()

content = soup.find("main") or soup.body or soup
website_text = content.get_text(separator=" ", strip=True)

if not website_text.strip():
    raise ValueError("Website lo readable text dorakaledhu.")

print("✅ Portfolio text loaded!")
print("\nPreview:\n", website_text[:1500])


✅ Portfolio text loaded!

Preview:
 Akula Hema Priya Hello, I’m Akula Hema Priya BTECH Student - Workin on Artificial intelligences About Me ``` Who I Am Hello! I’m Akula Hema Priya , a passionate and
        curious B.Tech Computer and Artificial Intelligence (CAI)
        student at Kakinada Institute of Engineering
        and Technologies (KIET) . I am interested in exploring the world of technology and
        understanding how computers, software, and Artificial Intelligence
        can be used to solve real-world problems. I enjoy learning new
        concepts and turning what I learn into practical projects. My areas of interest include Artificial Intelligence,
        Machine Learning, Web Development, App Development, and
        Programming . I am also interested in exploring new
        technologies and developing creative ideas that can make
        everyday tasks easier and more useful. As a student, I believe that skills grow through continuous
        practice and exper

In [19]:
words = website_text.split()

new_pieces = [
    {
        "file": url,
        "text": " ".join(words[start:start + 200])
    }
    for start in range(0, len(words), 150)
]

new_vectors = model.encode(
    [piece["text"] for piece in new_pieces],
    normalize_embeddings=True
)

PIECES = new_pieces
VECTORS = new_vectors

print("✅ Portfolio Q&A ready!")

resume_agent("What education is mentioned in this portfolio?")


✅ Portfolio Q&A ready!
QUESTION: What education is mentioned in this portfolio?
SOURCE: https://hemapriya-dev.github.io/hema-priya-portfolio/
SIMILARITY: 0.44
ANSWER: experience through projects and technical activities. I am a beginner in my professional journey, but I am always excited to learn, experiment, and take on new challenges. My goal is to become a skilled technology professional and create innovative applications that combine creativity with technology. My Interests Artificial Intelligence & Machine Learning Web & App Development Programming Data Analysis UI/UX Design Problem Solving Innovative Projects Learning New Technologies ``` Experience Frontend Developer (Intern) Swapna Bharathi Learning Solutions | 6 months duration – Completed Developed responsive web pages using HTML, CSS, and JavaScript. Improved UI design and usability. Collaborated with team members on frontend features. Education BTECH in CAI Kakinada Institute of Engineering Technology 2026 – 2029 Focused on